In [1]:
try:
    import jupytercloud.nirvana as jn
except:
    class jn:
        @staticmethod
        def get_named_input(name, local_path):
            return local_path
        
        @staticmethod
        def get_custom_output(index, local_path):
            from pathlib import Path
            return Path(local_path)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

NUM_SAMPLES = 20
# NUM_MODELS = 5 # deprecated, recommended at least 5 models tho
BOOTSTRAP_RATIO = 0.7

In [2]:
from statsmodels.stats.weightstats import zconfint, ztest
import yt.wrapper as yt

client = yt.YtClient('hahn')

def yt_read(path, client=client):
    result = yt.read_table(path, format='yson', client=client)
    df = pd.DataFrame(list(result))
    if df.empty:
        schema = yt.get_attribute(path, 'schema', client=client)
        return pd.DataFrame(columns=tuple(column['name'] for column in schema))
    else:
        return df
    
def local_read(path):
    return pd.read_csv(jn.get_named_input(path, local_path=path), sep='\t')

In [3]:
random_df = local_read('random.tsv')
policy_dfs = []
CUR_NUM_MODEL = 0
while True:
    try:
        policy_dfs.append(local_read(f'policy_{CUR_NUM_MODEL}.tsv'))
    except:
        print(f"Number of models {CUR_NUM_MODEL}")
        break
    CUR_NUM_MODEL+=1

Number of models 5


In [4]:
metric_names = list(random_df.columns)
for n in ["rid", "show", "priority_show", "candidate_num"]:
    metric_names.remove(n)
metric_names

['click', 'nsr']

In [5]:
def compute_policy_stats(random_df, policy_df):
    # Бёрём пересечение поклазов по политике и показов по рандому
    policy = random_df.merge(policy_df, on=['rid', 'show'], how='inner').drop('rid', axis=1).drop('priority_show', axis=1)

    # Нормируем метрики на количество кандидтов (чтобы экстраполировать пересечение с рандомом на полный размер рандома)
    for metric_name in metric_names:
        policy[metric_name] *= policy.candidate_num

    # Группируем по тритменту, считаем экстраполированные показы
    policy = policy.groupby('show').sum().rename({'candidate_num':'shows'}, axis=1)

    # Добавляем сумму
    policy.loc['overall'] = policy.sum(axis=0)
    
    # Считаем метрики удельно на экстраполированный показ
    for metric in metric_names:
        policy[f'{metric}_by_show'] = policy[metric] / policy.shows
    
    return policy

def compute_random_stats(random_df):
    # Удаляем candidate_num, добавляем shows=1 (тут использована замена, чтобы сохранить порядок колонок)
    policy = random_df.rename({'candidate_num':'shows'}, axis=1).drop('rid', axis=1).drop('priority_show', axis=1)
    policy.shows = 1
    
    # Группируем по тритменту
    policy = policy.groupby('show').sum()

    # Добавляем сумму
    policy.loc['overall'] = policy.sum(axis=0)

    # Считаем метрики удельно на показ
    for metric in metric_names:
        policy[f'{metric}_by_show'] = policy[metric] / policy.shows
    
    return policy

In [6]:
# Считаем усредненные метрики по декартову произведению моделей и выборок
policy_stats_overall = None
policy_stats_overall_with_mean_std = None
for policy_df in policy_dfs:
    for num_sample in range(NUM_SAMPLES):
        _, random_df_sample = train_test_split(random_df, test_size=BOOTSTRAP_RATIO)
        policy_stats = compute_policy_stats(random_df_sample, policy_df)
        
        if policy_stats_overall is None:
            policy_stats_overall =  pd.DataFrame(index = policy_stats.index, columns = policy_stats.columns)
            policy_stats_overall_with_mean_std =  pd.DataFrame(index = policy_stats.index, columns = policy_stats.columns)
            for col in policy_stats_overall.columns:
                policy_stats_overall[col] = policy_stats_overall[col].apply(lambda d: d if isinstance(d, list) else [])
        else:
            for index in policy_stats.index:
                if index not in policy_stats_overall.index:
                    policy_stats_overall.loc[index] = [[] for _ in range(len(policy_stats_overall.columns))]
                    policy_stats_overall_with_mean_std.loc[index] = [np.nan for _ in range(len(policy_stats_overall.columns))]
                 
        for row in policy_stats.iterrows():
            index = row[0]
            for col, value in row[1].items():
                policy_stats_overall.loc[index, col].append(policy_stats.loc[index, col])

for row in policy_stats_overall.iterrows():
    index = row[0]
    for col, value in row[1].items():
        policy_stats_overall_with_mean_std.loc[index, col] = (np.array(policy_stats_overall.loc[index, col]).mean(),
                                                np.array(policy_stats_overall.loc[index, col]).std())

policy_stats_overall_with_mean_std

,shows,click,nsr,click_by_show,nsr_by_show
show,,,,,
2195,"(3.3838383838383836, 1.1255263254290198)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)"
2266,"(606058.88, 777.0708755319556)","(3363.84, 53.15744914873174)","(3363.84, 53.15744914873174)","(0.005550359728857147, 8.792316616343728e-05)","(0.005550359728857147, 8.792316616343728e-05)"
2709,"(48807.56, 191.0703702827835)","(45.4, 5.635601121442149)","(45.4, 5.635601121442149)","(0.0009302088044678824, 0.00011558624726552869)","(0.0009302088044678824, 0.00011558624726552869)"
3548,"(461804.73, 560.9292086351004)","(953.68, 28.057754721288727)","(953.68, 28.057754721288727)","(0.0020651141229192376, 6.067151041656402e-05)","(0.0020651141229192376, 6.067151041656402e-05)"
5617,"(4.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)"
5866,"(296092.3, 593.0106997348362)","(290.55, 22.94836595489971)","(290.55, 22.94836595489971)","(0.0009812779093443203, 7.748471497620618e-05)","(0.0009812779093443203, 7.748471497620618e-05)"
5933,"(2.2637362637362637, 0.8097641066787051)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)"
6001,"(17.96, 5.580179208591781)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)","(0.0, 0.0)"
6106,"(596947.64, 718.7733651158757)","(3157.31, 61.28795885000577)","(3157.31, 61.28795885000577)","(0.005289091101788256, 0.00010251545326923848)","(0.005289091101788256, 0.00010251545326923848)"


In [7]:
random_stats = compute_random_stats(random_df)
random_stats

,shows,click,nsr,click_by_show,nsr_by_show
show,,,,,
2195,4.0,0.0,0.0,0.000000,0.000000
2266,302619.0,1711.0,1711.0,0.005654,0.005654
2709,35049.0,31.0,31.0,0.000884,0.000884
3548,292846.0,653.0,653.0,0.002230,0.002230
5617,1.0,0.0,0.0,0.000000,0.000000
5866,102556.0,74.0,74.0,0.000722,0.000722
5933,2.0,0.0,0.0,0.000000,0.000000
6001,5.0,0.0,0.0,0.000000,0.000000
6106,306356.0,1715.0,1715.0,0.005598,0.005598


In [8]:
priority_stats = compute_policy_stats(random_df, random_df[random_df.show == random_df.priority_show][['rid','show']])
priority_stats

,shows,click,nsr,click_by_show,nsr_by_show
show,,,,,
2195,3.0,0.0,0.0,0.000000,0.000000
2266,850828.0,4718.0,4718.0,0.005545,0.005545
2709,17870.0,14.0,14.0,0.000783,0.000783
3548,558237.0,1075.0,1075.0,0.001926,0.001926
5617,4.0,0.0,0.0,0.000000,0.000000
5866,17011.0,0.0,0.0,0.000000,0.000000
5933,1.0,0.0,0.0,0.000000,0.000000
6106,838814.0,4431.0,4431.0,0.005282,0.005282
6217,1040.0,3.0,3.0,0.002885,0.002885


In [9]:
print('Uplift from random')
for metric in metric_names:
    uplift = policy_stats_overall_with_mean_std[f'{metric}_by_show']['overall'][0] / random_stats[f'{metric}_by_show']['overall']
    print(f'{metric}:\t{uplift}')
print()

print('Uplift from priority')
for metric in metric_names:
    uplift = policy_stats_overall_with_mean_std[f'{metric}_by_show']['overall'][0] / priority_stats[f'{metric}_by_show']['overall']
    print(f'{metric}:\t{uplift}')
print()

print('shows by random:              ' + str(random_stats['shows']['overall']))
print('extrapolated shows by policy: ' + str(policy_stats_overall_with_mean_std['shows']['overall'][0] / BOOTSTRAP_RATIO))

Uplift from random
click:	1.0180374213734953
nsr:	1.0180374213734953

Uplift from priority
click:	0.7176314912161912
nsr:	0.7176314912161912

shows by random:              1690990.0
extrapolated shows by policy: 4591106.214285715
